<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [10]:
!pip install peft transformers[torch] trl bitsandbytes datasets wandb -U --q

In [11]:
import os
import random, math
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

## Wandb

In [12]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [13]:
import wandb
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [14]:
os.environ["WANDB_PROJECT"] = "sft_nlp_lab"  # name your W&B project
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # log all model checkpoints

In [15]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [16]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [40]:
shuffled_dataset = dataset.shuffle(seed=42)
split_ds_1 = shuffled_dataset.train_test_split(test_size=0.1)
split_ds_2 = split_ds_1['test'].train_test_split(test_size=0.5)

train_dataset = split_ds_1["train"].select(range(3000))
eval_dataset = split_ds_2["train"].select(range(2000))
test_dataset = split_ds_2["test"].select(range(10))

In [46]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


In [47]:
# HF hub ID
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [48]:
# Lab
lora_r = 16
lora_target_modules = "all-linear"
bs =  1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

# Others

In [49]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [50]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [51]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [52]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=16,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [53]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="epoch",
    bf16=True,
    optim=optim,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,

    # Scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Compute loss on model answers only
    completion_only_loss=True,

    # Logging
    logging_steps=50,
    report_to="wandb", # enables logging to W&B 😎
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,                # SFTConfig object
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [54]:
trainer.train()

Step,Training Loss
50,6.182000
100,5.585200
150,5.421100
200,5.197900
250,5.070200
300,4.991100
350,5.038300
400,4.957800
450,5.066000
500,4.955600


wandb: Adding directory to artifact (qlora-peft-output/checkpoint-750)... Done. 0.0s


TrainOutput(global_step=750, training_loss=5.141983479817708, metrics={'train_runtime': 85.7801, 'train_samples_per_second': 34.973, 'train_steps_per_second': 8.743, 'total_flos': 19294319221632.0, 'train_loss': 5.141983479817708, 'epoch': 1.0})

In [55]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [56]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: ./qlora-peft-output/adapter


In [57]:
!tree


.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-750
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth

In [58]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

Loading base model...
Loading adapter onto base model...
Merging LoRA adapter into base model weights...

Merged model saved to: ./merged-model



In [59]:
!tree

.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-750
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth

# Run merged-model inference

In [60]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [61]:
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"Output: {p['output']}\n")
    output = generate(wrapped)
    print(output)
    print("\n", "-" * 120, "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


==================== MERGED MODEL OUTPUT ====================

Output: ```python
import requests
import bs4

url = 'https://example.com/articles'

# Fetch the page
response = requests.get(url)
html_soup = bs4.BeautifulSoup(response.text, 'html.parser')

# Extract list of articles from the page
articles = html_soup.find_all('article')

# Iterate over the articles
for article in articles:
 # Extract the title, author, and content
 title = article.find('h2').text
 author = article.find('p', {'class': 'author'}).text
 content = article.find('p', {'class': 'content'}).text
 
 # Print out the extracted values
 print("Title:", title)
 print("Author:", author)
 print("Content:", content)
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a Python web scraper that extracts the title, author, and content of articles from a website The URL of a sample website to scrap is:
https://example.com/articles
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
import cv2
img = cv2.imread('image.jpg')
cv2.imshow('Image', img)
cv2.waitKey(0)
cv2.destroyAllWindows()
# Code executed.
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a web scraper in Python that scrapes the website "example.com" and prints out the content of all the h1 tags
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
import statistics

def descriptice_stats(numbers):
    print("Mean:", statistics.mean(numbers))
    print("Median:", statistics.median(numbers))
    print("Mode:", statistics.mode(numbers))
    print("Standard Deviation:", statistics.stdev(numbers))
    print("Variance:", statistics.variance(numbers))

if __name__ == '__main__':
    numbers = [2.3, 4.7, 5.2, 3.7]
    descriptice_stats(numbers)
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Write a Python program to generate descriptive statistics of a given list of numbers numbers = [2.3, 4.7, 5.2, 3.7]
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
maxnum = max(list1)
minnum = min(list1)
print("Maximum number is:", maxnum) 
print("Minimum number is:", minnum) 
#prints Maximum number is: 7   Minimum number is: 2
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a Python script to find the maximum and minimum numbers from a list list1 = [2, 3, 7, 4, 5]
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
for i in range(1,11):
   if (i % 2 != 0):
       print(i)
```

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Write a for loop to output all the odd numbers from 1 to 10 in Python.
Answer:




























































































































































































In [62]:
trainer.evaluate()

{'eval_loss': 4.960402965545654,
 'eval_runtime': 13.1623,
 'eval_samples_per_second': 151.949,
 'eval_steps_per_second': 18.994,
 'eval_entropy': 5.252371564865112,
 'eval_num_tokens': 464373.0,
 'eval_mean_token_accuracy': 0.2484570004940033,
 'epoch': 1.0}

In [63]:
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss']) if metrics['eval_loss'] < 20 else float('inf')
print('KL-SFT post-train eval_loss:', metrics['eval_loss'], 'ppl:', ppl)
print('Eval losses over steps:', [(h.get('step'), h.get('eval_loss')) for h in trainer.state.log_history if 'eval_loss' in h])


KL-SFT post-train eval_loss: 4.960402965545654 ppl: 142.65126786259293
Eval losses over steps: [(750, 4.960402965545654), (750, 4.960402965545654)]


# Resources

- https://wandb.ai/capecape/alpaca_ft/reports/How-to-Fine-tune-an-LLM-Part-3-The-HuggingFace-Trainer--Vmlldzo1OTEyNjMy
- https://youtu.be/t1caDsMzWBk